# Cat embeddings model
MobileNetV3-Large repurposed to create a lightweight embeddings model for cat behaviour monitoring

## To do
- ✅ Remove classification layers
- ✅ Add ONNX export path
- ✅ Demo embeddings using clustering


## Configure notebook

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir, "src")))

In [ ]:
from pathlib import Path

import numpy as np
import onnxruntime as ort
import torch
from matplotlib import pyplot as plt
from PIL import Image
from sklearn.cluster import KMeans
from torchvision import models

import utils

In [ ]:
%matplotlib inline

In [ ]:
device = utils.get_best_device()

## Load and prepare embeddings model

In [ ]:
# Load backbone model
imgsz = 320
weights = models.MobileNet_V3_Large_Weights.DEFAULT
preprocess = models.MobileNet_V3_Large_Weights.DEFAULT.transforms(
    crop_size=imgsz, resize_size=imgsz
)
backbone = models.mobilenet_v3_large(weights=weights)

# Keep feature extractor + global pooling only.
embedding_model = torch.nn.Sequential(
    backbone.features,
    backbone.avgpool,
    torch.nn.Flatten(1),
)
_ = embedding_model.to(device).eval()

## Quantise model

In [ ]:
quantized_model = embedding_model.to("cpu").half().eval()

onnx_dir = Path("..") / "models" / "mobilenetv3_large_embeddings_onnx_model"
onnx_dir.mkdir(parents=True, exist_ok=True)

dummy = torch.randn(1, 3, imgsz, imgsz, dtype=torch.float16)
onnx_path = onnx_dir / "model.onnx"

onnx_program = torch.onnx.export(
    quantized_model,
    (dummy,),
    input_names=["images"],
    output_names=["embeddings"],
    opset_version=18,
    dynamic_shapes=None,
)
onnx_program.save(onnx_path, external_data=False)

## Minimal inference helper

In [ ]:
ort_session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
ort_input_name = ort_session.get_inputs()[0].name

In [ ]:
x = np.random.rand(1, 3, imgsz, imgsz).astype(np.float16)
y = ort_session.run(None, {ort_input_name: x})[0]
print(y.shape)

In [ ]:
@torch.inference_mode()
def embed_image(image_np: np.ndarray) -> np.ndarray:
    if image_np.ndim != 3 or image_np.shape[2] != 3:
        raise ValueError("image_np must be an HxWx3 RGB array")

    if image_np.dtype != np.uint8:
        image_np = image_np.astype(np.uint8)

    tensor = preprocess(Image.fromarray(image_np)).unsqueeze(0).to("cpu").half()
    embedding = ort_session.run(None, {ort_input_name: tensor.numpy()})[0]
    embedding = torch.from_numpy(embedding).float()
    embedding = torch.nn.functional.normalize(embedding, p=2, dim=1)
    return embedding[0].cpu().numpy()

## Cluster sample images by embedding similarity

In [ ]:
sample_images = sorted((Path("..") / "sample_images").glob("*.jpg"))

embeddings = np.stack(
    [embed_image(np.asarray(Image.open(path).convert("RGB"))) for path in sample_images]
)
n_clusters = min(3, len(sample_images))

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init="auto")
labels = kmeans.fit_predict(embeddings)

clusters = {cluster_id: [] for cluster_id in range(n_clusters)}
for path, label in zip(sample_images, labels):
    clusters[label].append(path)

In [ ]:
max_cols = max(len(paths) for paths in clusters.values())
fig, axes = plt.subplots(
    n_clusters, max_cols, figsize=(3 * max_cols, 3 * n_clusters), squeeze=False
)

for row in range(n_clusters):
    paths = clusters[row]
    for col in range(max_cols):
        ax = axes[row][col]
        ax.axis("off")
        if col < len(paths):
            image = np.asarray(Image.open(paths[col]).convert("RGB"))
            ax.imshow(image)
            ax.set_title(f"C{row}: {paths[col].name}", fontsize=9)

fig.tight_layout()